In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path
import flammkuchen as fl

In [ ]:
class SessionAveragedDecoder:
    def __init__(self, hidden_layer_sizes=(100, 50), random_state=42):
        self.scaler = StandardScaler()
        self.classifier = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            random_state=random_state,
            max_iter=1000
        )
        
    def select_reliable_neurons(self, templates, selection_fraction=0.75):
        """
        Select reliable neurons using simple criteria.
        
        Parameters:
        templates: array of shape (n_neurons, n_positions, response_window_size)
        selection_fraction: fraction of neurons to keep (between 0 and 1)
                          e.g., 0.75 means keep top 75% of neurons
        
        Returns:
        reliable_mask: boolean array indicating reliable neurons
        reliability_scores: dictionary of scores for each neuron
        """
        # Ensure selection_fraction is between 0 and 1
        selection_fraction = np.clip(selection_fraction, 0.01, 1.0)
        n_neurons = templates.shape[0]
        reliability_scores = {}
        
        # Calculate mean response for each position
        mean_by_pos = np.mean(templates, axis=2)  # Average over time
        
        # Calculate position selectivity
        pos_selectivity = np.max(mean_by_pos, axis=1) - np.min(mean_by_pos, axis=1)
        reliability_scores['selectivity'] = pos_selectivity
        
        # Calculate response amplitude
        mean_response = np.mean(np.abs(templates), axis=(1,2))
        reliability_scores['amplitude'] = mean_response
        
        # Scale metrics to 0-1 range
        selectivity_scaled = (pos_selectivity - np.min(pos_selectivity)) / (np.max(pos_selectivity) - np.min(pos_selectivity) + 1e-6)
        amplitude_scaled = (mean_response - np.min(mean_response)) / (np.max(mean_response) - np.min(mean_response) + 1e-6)
        
        # Combine metrics
        overall_score = (selectivity_scaled + amplitude_scaled) / 2
        reliability_scores['overall'] = overall_score
        
        # Select neurons
        n_select = max(2, int(n_neurons * selection_fraction))
        top_indices = np.argsort(overall_score)[-n_select:]
        reliable_mask = np.zeros(n_neurons, dtype=bool)
        reliable_mask[top_indices] = True
        
        return reliable_mask, reliability_scores
    
    def plot_reliability_analysis(self, templates, reliability_scores, reliable_mask):
        """Plot reliability analysis results."""
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # Selectivity distribution
        sns.histplot(reliability_scores['selectivity'], ax=axes[0,0])
        axes[0,0].axvline(reliability_scores['selectivity'][reliable_mask].min(), 
                         color='r', linestyle='--', label='Selection threshold')
        axes[0,0].set_title('Position Selectivity Distribution')
        axes[0,0].legend()
        
        # Response amplitude distribution
        sns.histplot(reliability_scores['amplitude'], ax=axes[0,1])
        axes[0,1].axvline(reliability_scores['amplitude'][reliable_mask].min(), 
                         color='r', linestyle='--', label='Selection threshold')
        axes[0,1].set_title('Response Amplitude Distribution')
        axes[0,1].legend()
        
        # Example neurons
        if np.any(reliable_mask):
            reliable_idx = np.where(reliable_mask)[0][0]
            axes[1,0].set_title(f'Example Selected Neuron (ID: {reliable_idx})')
            for pos in range(templates.shape[1]):
                axes[1,0].plot(templates[reliable_idx, pos], 
                              label=f'Position {pos}')
            axes[1,0].legend()
        
        if np.any(~reliable_mask):
            unreliable_idx = np.where(~reliable_mask)[0][0]
            axes[1,1].set_title(f'Example Non-selected Neuron (ID: {unreliable_idx})')
            for pos in range(templates.shape[1]):
                axes[1,1].plot(templates[unreliable_idx, pos], 
                              label=f'Position {pos}')
            axes[1,1].legend()
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        print(f"\nReliability Analysis Summary:")
        print(f"Total neurons: {len(reliable_mask)}")
        print(f"Selected neurons: {np.sum(reliable_mask)} ({np.mean(reliable_mask)*100:.1f}%)")
        print(f"Average selectivity - Selected: {np.mean(reliability_scores['selectivity'][reliable_mask]):.3f}")
        print(f"Average selectivity - Non-selected: {np.mean(reliability_scores['selectivity'][~reliable_mask]):.3f}")
        print(f"Average amplitude - Selected: {np.mean(reliability_scores['amplitude'][reliable_mask]):.3f}")
        print(f"Average amplitude - Non-selected: {np.mean(reliability_scores['amplitude'][~reliable_mask]):.3f}")
    
    def create_response_templates(self, neural_data_list, position_matrix_list, 
                                response_window=(5, 15)):
        """
        Create averaged response templates for each neuron and stimulus position.
        
        Parameters:
        neural_data_list: list of arrays, each of shape (n_timepoints, n_neurons_in_group)
        position_matrix_list: list of arrays, each of shape (n_positions, n_timepoints)
        response_window: tuple (start, end) frames relative to stimulus onset
        
        Returns:
        templates: array of shape (n_total_neurons, n_positions, response_window_size)
        """
        n_positions = position_matrix_list[0].shape[0]  # Number of positions
        window_size = response_window[1] - response_window[0]
        
        # Calculate total number of neurons
        n_total_neurons = sum(data.shape[1] for data in neural_data_list)
        
        # Initialize arrays for sum and count of responses
        response_sum = np.zeros((n_total_neurons, n_positions, window_size))
        response_count = np.zeros((n_total_neurons, n_positions))
        
        # Keep track of neuron indexing across groups
        neuron_offset = 0
        
        # Process each group
        for neural_data, position_matrix in zip(neural_data_list, position_matrix_list):
            n_timepoints, n_neurons_in_group = neural_data.shape
            
            # Find stimulus onsets for each position
            for pos in range(n_positions):
                pos_vector = position_matrix[pos]
                onset_times = np.where(np.diff(pos_vector) == 1)[0] + 1
                
                for onset in onset_times:
                    if onset + response_window[1] <= n_timepoints:
                        response = neural_data[onset + response_window[0]:
                                             onset + response_window[1]]
                        
                        # Add to running sum for each neuron in this group
                        response_sum[neuron_offset:neuron_offset + n_neurons_in_group, 
                                   pos] += response.T
                        response_count[neuron_offset:neuron_offset + n_neurons_in_group, 
                                     pos] += 1
            
            neuron_offset += n_neurons_in_group
        
        # Calculate averages, handling divide by zero
        with np.errstate(divide='ignore', invalid='ignore'):
            templates = response_sum / response_count[:, :, np.newaxis]
        templates = np.nan_to_num(templates)  # Replace NaN with 0
        
        return templates
    
    def prepare_data_from_templates(self, templates, train_fraction=0.8):
        """
        Prepare training and testing data from templates.
        
        Parameters:
        templates: array of shape (n_neurons, n_positions, response_window_size)
        train_fraction: fraction of neurons to use for training
        
        Returns:
        X_train, X_test, y_train, y_test
        """
        n_neurons, n_positions, window_size = templates.shape
        print(f"Template shape: {templates.shape}")
        
        # Reshape templates to have one sample per neuron-position combination
        X = templates.reshape(n_neurons * n_positions, window_size)
        print(f"Reshaped X shape: {X.shape}")
        
        # Create labels (one for each neuron-position combination)
        y = np.repeat(np.arange(n_positions), n_neurons)
        print(f"y shape: {y.shape}")
        print(f"Unique y values: {np.unique(y)}")
        
        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            train_size=train_fraction,
            random_state=42,
            stratify=y  # Ensure balanced split across positions
        )
        
        print(f"X_train shape: {X_train.shape}")
        print(f"X_test shape: {X_test.shape}")
        print(f"y_train shape: {y_train.shape}")
        print(f"y_test shape: {y_test.shape}")
        
        # Scale the data
        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)
        
        return X_train, X_test, y_train, y_test
    
    
    def train(self, X_train, y_train):
        """Train the decoder."""
        print(f"Training data shapes - X: {X_train.shape}, y: {y_train.shape}")
        self.classifier.fit(X_train, y_train)
    
    def predict(self, X):
        """Make predictions."""
        return self.classifier.predict(self.scaler.transform(X))
    
    def evaluate(self, X_test, y_test):
        """Evaluate decoder performance."""
        y_pred = self.predict(X_test)
        accuracy = self.classifier.score(self.scaler.transform(X_test), y_test)
        conf_mat = confusion_matrix(y_test, y_pred)
        return accuracy, conf_mat
    
    def plot_confusion_matrix(self, conf_mat):
        """Plot the confusion matrix."""
        plt.figure(figsize=(10, 8))
        sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues')
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted Position')
        plt.ylabel('True Position')
        plt.show()

def demo_session_averaged_decoder(neural_data_list, position_matrix_list, 
                                response_window=(5, 15), train_fraction=0.8,
                                selection_fraction=0.75):
    """
    Demonstrate the session-averaged decoder with neuron selection.
    
    Parameters:
    neural_data_list: list of arrays, each of shape (n_timepoints, n_neurons_in_group)
    position_matrix_list: list of arrays, each of shape (n_positions, n_timepoints)
    response_window: tuple (start, end) frames for response window
    train_fraction: fraction of neurons to use for training
    selection_fraction: fraction of neurons to keep (between 0 and 1)
    """
    # Initialize decoder
    decoder = SessionAveragedDecoder()
    
    # Create response templates
    print("Creating response templates...")
    templates = decoder.create_response_templates(
        neural_data_list,
        position_matrix_list,
        response_window
    )
    
    # Select reliable neurons
    print("\nSelecting reliable neurons...")
    reliable_mask, reliability_scores = decoder.select_reliable_neurons(
        templates,
        selection_fraction=selection_fraction
    )
    
    # Plot reliability analysis
    decoder.plot_reliability_analysis(templates, reliability_scores, reliable_mask)
    
    # Use only reliable neurons
    templates_reliable = templates[reliable_mask]
    
    # Prepare data using reliable neurons only
    print("\nPreparing data with reliable neurons...")
    X_train, X_test, y_train, y_test = decoder.prepare_data_from_templates(
        templates_reliable,
        train_fraction
    )
    
    # Train decoder
    print("Training decoder...")
    decoder.train(X_train, y_train)
    
    # Evaluate
    accuracy, conf_mat = decoder.evaluate(X_test, y_test)
    print(f"\nTest set accuracy: {accuracy:.2f}")
    
    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (Reliable Neurons Only)')
    plt.xlabel('Predicted Position')
    plt.ylabel('True Position')
    plt.show()
    
    return decoder, templates, reliable_mask, reliability_scores

In [ ]:
master =  Path(r"Z:\Hagar and Ot\e0075\habenula")
fish_list = list(master.glob("*_f*"))

In [ ]:
fish = fish_list[-4] / "suite2p"
planes = list(fish.glob("*00*"))
    
neural_data_groups_l = []
neural_data_groups_r = []
stimulus_data_groups = []

for plane in planes:
    
    traces = fl.load(plane / 'filtered_traces.h5')['undetr']
    regs = fl.load(plane / 'sensory_regressors_cells.h5')['regressors']

    hab_coords_l = fl.load(plane / 'habenula_coords.h5')['lhab_coords']
    hab_coords_r = fl.load(plane / 'habenula_coords.h5')['rhab_coords']

    habenula_traces_l = traces[:,hab_coords_l]
    habenula_traces_r = traces[:,hab_coords_r]
    
    neural_data_groups_l =  neural_data_groups_l + [habenula_traces_l]
    neural_data_groups_r =  neural_data_groups_r + [habenula_traces_r]
    
    stimulus_data_groups = stimulus_data_groups + [regs]

In [ ]:
decoder, templates, reliable_mask, reliability_scores = demo_session_averaged_decoder(
    neural_data_groups_l, 
    stimulus_data_groups,
    selection_fraction=0.75,  # Adjust this to be more/less selective
)